# Sample images from LILA BC

1. Get image URLs and labels from LILA BC
2. Sample n images for each class
3. Train val test split
4. Save CSV with sampled image URLs to Drive

## Setup

In [ ]:
import os
import shutil
import sys

import polars as pl

from google.colab import drive
drive.mount("/content/drive")

# Check if repo exists in Drive
drive_repo_path = "/content/drive/MyDrive/TeraiNet"
if os.path.exists(drive_repo_path):
    project_dir = drive_repo_path
    print(f"Using existing repo from: {project_dir}")
else:
    # Clone from GitHub to Drive
    print("Repo not found in Drive. Cloning from GitHub...")
    !git clone https://github.com/alexvmt/terainet.git "$drive_repo_path"
    project_dir = drive_repo_path

# Install package in editable mode
print("Installing TeraiNet package...")
!pip install -e "$project_dir"

from terainet import (
    add_subset_column,
    check_location_split,
    load_config,
    sample_n_images_per_species,
)

In [ ]:
# load config and set variables and parameters
config_path = os.path.join(project_dir, "config.yaml")
config = load_config(config_path)

scripts_dir = os.path.join(project_dir, config["sampling_downloading"]["scripts_dir"])

samples_dir = os.path.join(project_dir, config["sampling_downloading"]["samples_dir"])

train_ratio = config["training"]["train_ratio"]

In [ ]:
# for a fresh start, remove samples dir
remove_samples_dir = True
if remove_samples_dir:
    !rm -rf "$samples_dir"
    !mkdir -p "$samples_dir"

## Download image URLs and labels from LILA BC

In [ ]:
!wget -O lila_image_urls_and_labels.csv.zip -nc "https://lila.science/public/lila_image_urls_and_labels.csv.zip"

In [ ]:
![ -f lila_image_urls_and_labels.csv ] || unzip lila_image_urls_and_labels.csv.zip

In [ ]:
urls_and_labels = "lila_image_urls_and_labels.csv"

## Inspect species counts

Check taxonomy mapping to find relevant species:
https://lila.science/public/lila-taxonomy-mapping_release.csv

In [ ]:
columns = [
    "url_gcp",
    "image_id",
    "sequence_id",
    "location_id",
    "frame_num",
    "datetime",
    "common_name",
]

schema_overrides = {
    "url_gcp": pl.Utf8(),
    "image_id": pl.Utf8(),
    "sequence_id": pl.Utf8(),
    "location_id": pl.Utf8(),
    "frame_num": pl.Int32(),
    "datetime": pl.Utf8(),
    "common_name": pl.Utf8(),
}

lila_image_urls_and_labels_df = pl.read_csv(
    "lila_image_urls_and_labels.csv", columns=columns, schema_overrides=schema_overrides
)
lila_image_urls_and_labels_df.shape

In [ ]:
lila_image_urls_and_labels_df.head()

In [ ]:
# TODO: figure out why common_name is empty in half the rows and what this implies
lila_image_urls_and_labels_df.filter(pl.col("common_name").is_null()).shape

In [ ]:
species_list = [
    "tiger",
    "leopard",
    "asian black bear",
    "american black bear",  # not enough images of asian black bear alone
    "dhole",
    "black-backed jackal",
    "gray fox",
    "leopard cat",
    "mainland leopard cat",
    "marbled cat",
    "asian golden cat",  # other carnivores (including substitutes, i. e. black-backed jackal and gray fox)
    "deer",
    "wild boar",
    "african buffalo",
    "cape buffalo",  # substitute for gaur
    "white rhinoceros",  # substitute for indian rhino
    "asian elephant",
    "african bush elephant",  # not enough images of asian elephant alone
    "bird",
]

In [ ]:
lila_image_urls_and_labels_df = lila_image_urls_and_labels_df.filter(
    pl.col("common_name").is_in(species_list)
)
species_stats = lila_image_urls_and_labels_df.group_by("common_name").agg(
    pl.len().alias("total_count"), pl.n_unique("location_id").alias("unique_location_count")
)
pl.Config.set_tbl_rows(species_stats.shape[0])
species_stats.sort("total_count")

## Get image URLs from LILA BC

The goal is to get about 3000 images per class (+100 buffer). Leopard images availability sets this limit because we want classes to be balanced. For tiger images, which are even scarcer, there fortunately is another additional source (Amur tiger re-identification challenge)

### Tiger

In [ ]:
class_name = "tiger"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["tiger"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
# use all tiger images because we don't have many
species_samples_dict = {"tiger": all_image_urls.shape[0]}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
# put all lila bc tiger images into separate test set to test how well model trained on amur tiger images generalizes
sampled_image_urls = sampled_image_urls.with_columns(
    pl.lit(class_number).alias("class_number"), pl.lit("test2").alias("subset")
)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"

### Leopard

In [ ]:
class_name = "leopard"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["leopard"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
# use all leopard images because because we don't have many
species_samples_dict = {"leopard": 2991}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

### Black bear

Include `american black bear` because there are not enough camera trap images of Asian black bears.

In [ ]:
class_name = "black_bear"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["asian black bear", "american black bear"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
species_samples_dict = {"asian black bear": 1221, "american black bear": 1879}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

### Other carnivores

Include `dhole,black-backed jackal,gray fox,leopard cat,mainland leopard cat,marbled cat,asian golden cat` to cover a wide range of other carnivores in the Terai ecosystem.

In [ ]:
class_name = "other_carnivores"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = [
    "dhole",
    "black-backed jackal",
    "gray fox",
    "leopard cat",
    "mainland leopard cat",
    "marbled cat",
    "asian golden cat",
]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
species_samples_dict = {
    "dhole": 185,
    "black-backed jackal": 890,
    "gray fox": 890,
    "leopard cat": 266,
    "mainland leopard cat": 246,
    "marbled cat": 271,
    "asian golden cat": 353,
}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

### Deer

In [ ]:
class_name = "deer"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["deer"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
species_samples_dict = {"deer": 3100}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

### Wild boar

In [ ]:
class_name = "wild_boar"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["wild boar"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
species_samples_dict = {"wild boar": 3100}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

### Buffalo

Use `african buffalo,cape buffalo` because camera trap images of gaur are unavailable.

In [ ]:
class_name = "buffalo"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["african buffalo", "cape buffalo"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
species_samples_dict = {"african buffalo": 1550, "cape buffalo": 1550}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

### Rhino

Use `white rhinoceros` because camera trap images of Indian rhinoceros are unavailable.

In [ ]:
class_name = "rhino"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["white rhinoceros"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
species_samples_dict = {"white rhinoceros": 3100}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

### Elephant

Include `african bush elephant` because there are not enough camera trap images of Asian elephants.

In [ ]:
class_name = "elephant"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["asian elephant", "african bush elephant"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
species_samples_dict = {"asian elephant": 325, "african bush elephant": 2775}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)

### Bird

In [ ]:
class_name = "bird"
class_number = config["classes"][class_name]
column_to_filter = "common_name"
values_to_filter = ["bird"]
all_image_urls = lila_image_urls_and_labels_df.filter(
    pl.col(column_to_filter).is_in(values_to_filter)
)
print(f"All rows: {all_image_urls.shape[0]}")

In [ ]:
species_samples_dict = {"bird": 3100}

sampled_image_urls = sample_n_images_per_species(
    all_image_urls, species_samples_dict, column_to_filter
)
sampled_image_urls = sampled_image_urls.with_columns(pl.lit(class_number).alias("class_number"))
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_image_urls_filename = os.path.join(
    samples_dir,
    "lila_bc_image_urls_" + class_name + "_sampled_class_number_" + str(class_number) + ".csv",
)
sampled_image_urls.write_csv(sampled_image_urls_filename, separator=",")
print(f"ℹ️ Species distribution:\n{sampled_image_urls[column_to_filter].value_counts()}")
!echo "ℹ️ Total sampled rows: $(tail -n +2 "$sampled_image_urls_filename" | wc -l)"
check_location_split(sampled_image_urls)